# P-05: Entities should have an `organisation_entity`

**Author**: Sian Wilson <br>
**Date created**: 24th September 2026 <br>
**Dataset Scope**: all datasets on the platform with a published Datasette database <br>
**Purpose**: Ticket P-05 — check how prevalent it is for entities in a dataset's `entity` table to have no value in `organisation_entity` at all. <br>

`analysis/2026-07_correct_organisations_of_entities/` was raised as possibly-related prior work, but that notebook looks at a *different* population: entities that **do** have an `organisation_entity`, but possibly the wrong one. It explicitly drops any entity with a blank `organisation_entity` before its trace even starts, and never counts how many that was. This notebook measures that population directly, across every dataset on the platform, regardless of `quality`.

If this turns out to be widespread, next step is a conversation with Swati about how to tackle it (one-off data fix vs a new pipeline task) — not covered here.

In [1]:
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd

from helpers import datasette_sql, list_published_datasets, fetch_table_csv, fetch_config_csv, list_config_pipelines_with_file, chunk

DATA_DIR = os.path.join("..", "..", "data")
os.makedirs(DATA_DIR, exist_ok=True)

## 1. List datasets to check

`dataset` (in the `digital-land` db) lists every registered dataset, but most of those have no published Datasette database at all -- nothing has been collected for them yet, which is a normal platform state, not a fetch failure. Only datasets with an actual published database can be checked here.

In [2]:
all_datasets = datasette_sql("digital-land", "SELECT dataset FROM dataset ORDER BY dataset")["dataset"].tolist()
published = list_published_datasets()
datasets_to_check = sorted(d for d in all_datasets if d in published)

print(f"{len(all_datasets)} registered datasets, {len(published)} have a published database")
print(f"{len(datasets_to_check)} datasets will be checked; {len(all_datasets) - len(datasets_to_check)} skipped (no data collected on the platform yet)")

281 registered datasets, 126 have a published database
124 datasets will be checked; 157 skipped (no data collected on the platform yet)


## 2. Per-dataset entity counts: total vs missing `organisation_entity`

One aggregate SQL query per dataset (not a row-level pull — some datasets have 100k+ entities and we only need counts here). Reuses the same NULL/blank guard already used elsewhere in this repo for `organisation_entity` (`reports/measure_data_quality/functions_import.py`). A handful of published datasets still don't have an `entity` table (e.g. non-spatial reference datasets) — those failures are reported explicitly below rather than silently folded into the "no data" group above, since they're a different, worth-checking case.

In [3]:
def _counts_for_dataset(dataset):
    # Two separate simple COUNT(*) queries, not one combined SUM(CASE...) aggregate --
    # the combined form hit Datasette's SQL time limit on large tables (e.g.
    # flood-risk-zone, 780k rows), while each COUNT on its own runs in seconds.
    total = datasette_sql(dataset, "SELECT COUNT(*) AS n FROM entity")["n"][0]
    missing = datasette_sql(
        dataset,
        "SELECT COUNT(*) AS n FROM entity WHERE organisation_entity IS NULL OR organisation_entity = ''",
    )["n"][0]
    return pd.DataFrame([{"dataset": dataset, "total_entities": total, "missing_org_entity": missing}])


print(f"Querying entity counts across {len(datasets_to_check)} datasets (parallel)...")
count_parts = []
failed_datasets = []
with ThreadPoolExecutor(max_workers=15) as pool:
    futures = {pool.submit(_counts_for_dataset, ds): ds for ds in datasets_to_check}
    for f in as_completed(futures):
        ds = futures[f]
        try:
            count_parts.append(f.result())
        except Exception as e:
            failed_datasets.append((ds, str(e)[:200]))

counts_df = pd.concat(count_parts, ignore_index=True)
counts_df["missing_org_entity"] = counts_df["missing_org_entity"].fillna(0).astype(int)
print(f"{len(counts_df)} datasets successfully checked")
if failed_datasets:
    print(f"{len(failed_datasets)} published datasets could NOT be checked (genuine failures, not 'no data'):")
    for ds, err in failed_datasets:
        print(f"  {ds}: {err}")

Querying entity counts across 124 datasets (parallel)...


122 datasets successfully checked
2 published datasets could NOT be checked (genuine failures, not 'no data'):
  title-boundary: Datasette SQL error (title-boundary): no such table: entity
  planning-application: Datasette SQL error (planning-application): <p>SQL query took too long. The time limit is controlled by the
<a href="https://docs.datasette.io/en/stable/settings.html#sql-time-limit-ms">sql_time_limit


## 3. Breakdown by `quality`, within the missing-`organisation_entity` subset

Shows whether blank-`organisation_entity` rows cluster in a particular `quality` value, and whether this is the same or a distinct population from the `quality='some'` misattribution issue already investigated in `2026-07_correct_organisations_of_entities`.

In [4]:
def _quality_breakdown_for_dataset(dataset):
    df = datasette_sql(dataset, """
        SELECT quality, COUNT(*) AS n
        FROM entity
        WHERE organisation_entity IS NULL OR organisation_entity = ''
        GROUP BY quality
    """)
    if df.empty:
        return None
    df["dataset"] = dataset
    return df


quality_parts = []
with ThreadPoolExecutor(max_workers=15) as pool:
    futures = {pool.submit(_quality_breakdown_for_dataset, ds): ds for ds in datasets_to_check}
    for f in as_completed(futures):
        try:
            r = f.result()
        except Exception:
            continue  # already reported in the cell above
        if r is not None:
            quality_parts.append(r)

quality_df = pd.concat(quality_parts, ignore_index=True) if quality_parts else pd.DataFrame(columns=["dataset", "quality", "n"])

print("Missing organisation_entity rows, by quality value (summed across all datasets):")
quality_df.groupby("quality")["n"].sum().sort_values(ascending=False)

Missing organisation_entity rows, by quality value (summed across all datasets):


quality
some             107
authoritative      1
Name: n, dtype: int64

## 4. Summary table

Sorted by `missing_pct` descending to surface the worst-offending datasets first.

In [5]:
summary_df = counts_df.copy()
summary_df["missing_pct"] = (summary_df["missing_org_entity"] / summary_df["total_entities"] * 100).round(1)
summary_df = summary_df[summary_df["missing_org_entity"] > 0].sort_values(
    ["missing_pct", "missing_org_entity"], ascending=False
)[["dataset", "total_entities", "missing_org_entity", "missing_pct"]]

print(f"{len(summary_df)} / {len(counts_df)} checked datasets have at least one entity missing organisation_entity")
print(f"Total entities missing organisation_entity across the platform: {summary_df['missing_org_entity'].sum()}")
summary_df

4 / 122 checked datasets have at least one entity missing organisation_entity
Total entities missing organisation_entity across the platform: 108


,dataset,total_entities,missing_org_entity,missing_pct
15,central-activities-zone,10,10,100.0
71,local-plan-timetable,2700,82,3.0
63,local-plan-document,509,8,1.6
11,brownfield-land,37736,8,0.0


## 5. Row-level entities missing `organisation_entity`

The summary table above is dataset-level counts. This pulls the actual affected entities -- one row per entity, with the dataset it belongs to -- for the handful of datasets found to have any, so they can be looked at/actioned directly rather than just counted.

In [6]:
def _missing_entities_for_dataset(dataset):
    df = datasette_sql(dataset, """
        SELECT entity, name, reference, quality
        FROM entity
        WHERE organisation_entity IS NULL OR organisation_entity = ''
        ORDER BY entity
    """)
    df["dataset"] = dataset
    return df


affected_datasets = summary_df["dataset"].tolist()
entity_parts = [_missing_entities_for_dataset(ds) for ds in affected_datasets]
missing_entities_df = pd.concat(entity_parts, ignore_index=True)
missing_entities_df["entity_url"] = "https://www.planning.data.gov.uk/entity/" + missing_entities_df["entity"].astype(str)
missing_entities_df = missing_entities_df[["dataset", "entity", "name", "reference", "quality", "entity_url"]]

print(f"{len(missing_entities_df)} entities missing organisation_entity across {missing_entities_df['dataset'].nunique()} datasets")
missing_entities_df

108 entities missing organisation_entity across 4 datasets


,dataset,entity,name,reference,quality,entity_url
0,central-activities-zone,2200001,,CAZ00000001,some,https://www.planning.data.gov.uk/entity/2200001
1,central-activities-zone,2200002,,CAZ00000002,some,https://www.planning.data.gov.uk/entity/2200002
2,central-activities-zone,2200003,,CAZ00000003,some,https://www.planning.data.gov.uk/entity/2200003
3,central-activities-zone,2200004,,CAZ00000004,some,https://www.planning.data.gov.uk/entity/2200004
4,central-activities-zone,2200005,,CAZ00000005,some,https://www.planning.data.gov.uk/entity/2200005
...,...,...,...,...,...,...
103,brownfield-land,1741629,7660,7660,some,https://www.planning.data.gov.uk/entity/1741629
104,brownfield-land,1741630,7713,7713,some,https://www.planning.data.gov.uk/entity/1741630
105,brownfield-land,1741642,BLR25001,BLR25001,some,https://www.planning.data.gov.uk/entity/1741642
106,brownfield-land,1741643,BLR25002,BLR25002,some,https://www.planning.data.gov.uk/entity/1741643


## 6. Full config-repo audit: `entity-organisation.csv` and `lookup.csv`, every dataset

Checking only the datasets already known to have a live blank `organisation_entity` (`central-activities-zone`, `brownfield-land`) isn't a complete check of the config-repo bug pattern itself -- a broken/duplicate range assignment might not always surface as blank on the live platform (it could coincidentally resolve to a valid-but-wrong row instead of blanking out). So this audits every pipeline in `digital-land/config` that has an `entity-organisation.csv` or `lookup.csv`, independent of what's currently blank live, and flags:

- **empty** -- a row exists but the `organisation` value itself is blank (only checked for `entity-organisation.csv`; a blank `organisation` in `lookup.csv` is the normal case for most rows there, not a fault).
- **invalid** -- a non-blank `organisation` curie that doesn't match anything in the live `organisation` table.

In [7]:
import difflib

org_df = fetch_table_csv("organisation")
valid_curies = set(org_df["organisation"].dropna())
curie_to_website = org_df.set_index("organisation")["website"].dropna().to_dict()


def _suggest_correction(curie, other_candidates_for_entity=()):
    # try a straight prefix swap to local-authority: first (the central-activities-zone pattern)
    if ":" in curie:
        code = curie.split(":", 1)[1]
        swapped = f"local-authority:{code}"
        if swapped in valid_curies:
            return swapped, "prefix swap to local-authority:"
    # next, prefer a near-identical match among this entity's OTHER config candidates over a
    # blind national search -- a duplicate/conflicting range (like brownfield-land's SAN/SAW)
    # is much more likely to be "the other row for this same entity" than a coincidentally
    # similar but unrelated council elsewhere (a blind search matched SAN -> TAN, Tandridge --
    # a different region entirely -- when SAW, Sandwell, was sitting right there as the other
    # candidate for the same entity).
    valid_siblings = [c for c in other_candidates_for_entity if c in valid_curies and c != curie]
    close_sibling = difflib.get_close_matches(curie, valid_siblings, n=1, cutoff=0.6)
    if close_sibling:
        return close_sibling[0], "closest match among this entity's other config candidates"
    # otherwise look for a near-identical valid curie nationally (typos, transpositions)
    close = difflib.get_close_matches(curie, valid_curies, n=1, cutoff=0.8)
    if close:
        return close[0], "closest match to a valid curie nationally"
    return None, None


eo_pipelines = list_config_pipelines_with_file("entity-organisation")
lookup_pipelines = list_config_pipelines_with_file("lookup")
print(f"{len(eo_pipelines)} pipelines have entity-organisation.csv, {len(lookup_pipelines)} have lookup.csv")

audit_rows = []

for dataset in eo_pipelines:
    eo_df = fetch_config_csv(dataset, "entity-organisation")
    if eo_df.empty:
        continue
    for _, row in eo_df.iterrows():
        org = row.get("organisation")
        org_str = str(org).strip() if pd.notna(org) else ""
        if org_str == "":
            audit_rows.append({
                "dataset": dataset, "file": "entity-organisation.csv", "issue_type": "empty",
                "organisation": None, "location": f"entity {row.get('entity-minimum')}-{row.get('entity-maximum')}",
                "suggested_fix": None, "confidence": None,
            })
        elif org_str not in valid_curies:
            # other candidate rows covering the same range, for the sibling-match heuristic
            same_range = eo_df[
                (eo_df["entity-minimum"] == row["entity-minimum"]) & (eo_df["entity-maximum"] == row["entity-maximum"])
            ]
            siblings = same_range["organisation"].dropna().unique().tolist()
            fix, method = _suggest_correction(org_str, siblings)
            audit_rows.append({
                "dataset": dataset, "file": "entity-organisation.csv", "issue_type": "invalid",
                "organisation": org_str, "location": f"entity {row.get('entity-minimum')}-{row.get('entity-maximum')}",
                "suggested_fix": f"{fix} ({method})" if fix else "no close match found",
                "confidence": "high" if fix else "low",
            })

for dataset in lookup_pipelines:
    lk_df = fetch_config_csv(dataset, "lookup")
    if lk_df.empty or "organisation" not in lk_df.columns:
        continue
    nonblank = lk_df[lk_df["organisation"].notna() & (lk_df["organisation"].astype(str).str.strip() != "")]
    invalid = nonblank[~nonblank["organisation"].isin(valid_curies)]
    for _, row in invalid.iterrows():
        fix, method = _suggest_correction(str(row["organisation"]))
        audit_rows.append({
            "dataset": dataset, "file": "lookup.csv", "issue_type": "invalid",
            "organisation": row["organisation"], "location": f"entity {row.get('entity')} (reference {row.get('reference')})",
            "suggested_fix": f"{fix} ({method})" if fix else "no close match found",
            "confidence": "high" if fix else "low",
        })

config_audit_df = pd.DataFrame(audit_rows)
print(f"\n{len(config_audit_df)} config issues found across {config_audit_df['dataset'].nunique() if not config_audit_df.empty else 0} datasets")
if not config_audit_df.empty:
    print(config_audit_df.groupby(["file", "issue_type"]).size())
    print()
    print(config_audit_df["dataset"].value_counts())
config_audit_df

55 pipelines have entity-organisation.csv, 55 have lookup.csv


/Users/sianteesdale/Documents/GitHub/jupyter-analysis/analysis/2026-09_data-quality-ticket-checks/helpers.py:48: DtypeWarning: Columns (1,4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(StringIO(r.text))



12 config issues found across 2 datasets
file                     issue_type
entity-organisation.csv  invalid       12
dtype: int64

dataset
central-activities-zone    10
brownfield-land             2
Name: count, dtype: int64


,dataset,file,issue_type,organisation,location,suggested_fix,confidence
0,brownfield-land,entity-organisation.csv,invalid,local-authority:SAN,entity 1741626-1741630,local-authority:SAW (closest match among this ...,high
1,brownfield-land,entity-organisation.csv,invalid,local-authority:WFK,entity 1741642-1741643,local-authority:WKF (closest match among this ...,high
2,central-activities-zone,entity-organisation.csv,invalid,local-organisation:CMD,entity 2200007-2200007,local-authority:CMD (prefix swap to local-auth...,high
3,central-activities-zone,entity-organisation.csv,invalid,local-organisation:HCK,entity 2200010-2200010,local-authority:HCK (prefix swap to local-auth...,high
4,central-activities-zone,entity-organisation.csv,invalid,local-organisation:ISL,entity 2200009-2200009,local-authority:ISL (prefix swap to local-auth...,high
5,central-activities-zone,entity-organisation.csv,invalid,local-organisation:KEC,entity 2200004-2200004,local-authority:KEC (prefix swap to local-auth...,high
6,central-activities-zone,entity-organisation.csv,invalid,local-organisation:LBH,entity 2200001-2200001,local-authority:LBH (prefix swap to local-auth...,high
7,central-activities-zone,entity-organisation.csv,invalid,local-organisation:LND,entity 2200006-2200006,local-authority:LND (prefix swap to local-auth...,high
8,central-activities-zone,entity-organisation.csv,invalid,local-organisation:SWK,entity 2200002-2200002,local-authority:SWK (prefix swap to local-auth...,high
9,central-activities-zone,entity-organisation.csv,invalid,local-organisation:TWH,entity 2200008-2200008,local-authority:TWH (prefix swap to local-auth...,high


## 7. Cross-reference: which live-blank entities does this explain?

Restricted to the two current (non-legacy) affected datasets -- `local-plan-document`/`local-plan-timetable` dropped as confirmed-stale (see section 5). For each of those entities, classify against the full config audit above:

- **`broken_mapping`** -- the entity's `entity-organisation.csv` range or `lookup.csv` row appears in the audit as `invalid`/`empty`.
- **`unexplained`** -- config assigns a single, valid curie, yet the entity is still blank live (not a config bug we can see -- likely a pipeline re-run/staleness issue).
- **`unassigned`** -- no config assignment found at all -- falls through to the URL-based suggestion in the next section.

In [8]:
affected_datasets = ["central-activities-zone", "brownfield-land"]

diagnosis_rows = []
for dataset in affected_datasets:
    entity_org_df = fetch_config_csv(dataset, "entity-organisation")
    if not entity_org_df.empty:
        entity_org_df["entity-minimum"] = pd.to_numeric(entity_org_df["entity-minimum"], errors="coerce")
        entity_org_df["entity-maximum"] = pd.to_numeric(entity_org_df["entity-maximum"], errors="coerce")
    lookup_df = fetch_config_csv(dataset, "lookup")
    if not lookup_df.empty:
        lookup_df["entity"] = pd.to_numeric(lookup_df["entity"], errors="coerce")

    dataset_entities = missing_entities_df[missing_entities_df["dataset"] == dataset]
    dataset_audit = config_audit_df[config_audit_df["dataset"] == dataset] if not config_audit_df.empty else pd.DataFrame()

    for _, row in dataset_entities.iterrows():
        entity = row["entity"]

        candidate_curies = []
        if not entity_org_df.empty:
            matches = entity_org_df[
                (entity_org_df["entity-minimum"] <= entity) & (entity_org_df["entity-maximum"] >= entity)
            ]
            candidate_curies += matches["organisation"].dropna().unique().tolist()
        if not lookup_df.empty:
            lookup_match = lookup_df[
                (lookup_df["entity"] == entity)
                & lookup_df["organisation"].notna()
                & (lookup_df["organisation"] != "")
            ]
            candidate_curies += lookup_match["organisation"].unique().tolist()
        candidate_curies = sorted(set(candidate_curies))

        valid_found = [c for c in candidate_curies if c in valid_curies]
        invalid_found = [c for c in candidate_curies if c not in valid_curies]

        suggested_curie = None
        if not candidate_curies:
            classification, suggested_fix, confidence = "unassigned", None, None
        elif invalid_found:
            classification = "broken_mapping"
            fixes = []
            resolved_any = bool(valid_found)
            first_fix = valid_found[0] if valid_found else None
            for bad in invalid_found:
                fix, method = _suggest_correction(bad, candidate_curies)
                if fix:
                    resolved_any = True
                    first_fix = first_fix or fix
                    fixes.append(f"{bad} -> {fix} ({method})")
                else:
                    fixes.append(f"{bad} -> no close match found")
            if valid_found:
                fixes.append(f"config also assigns valid {', '.join(valid_found)}")
            suggested_fix = "; ".join(fixes)
            suggested_curie = first_fix
            confidence = "high" if resolved_any else "low"
        elif len(candidate_curies) > 1:
            classification = "broken_mapping"
            suggested_fix = f"duplicate/conflicting valid assignments in config: {candidate_curies}"
            confidence = "medium"
        else:
            classification = "unexplained"
            suggested_fix = f"config already assigns valid {candidate_curies[0]}, but entity is still blank live -- likely needs a pipeline re-run, not a config fix"
            suggested_curie = candidate_curies[0]
            confidence = None

        diagnosis_rows.append({
            "dataset": dataset,
            "entity": entity,
            "name": row["name"],
            "reference": row["reference"],
            "classification": classification,
            "config_assigned_curies": "; ".join(candidate_curies) if candidate_curies else None,
            "suggested_fix": suggested_fix,
            "suggested_curie": suggested_curie,
            "confidence": confidence,
            "entity_url": row["entity_url"],
        })

diagnosis_df = pd.DataFrame(diagnosis_rows)
print(diagnosis_df["classification"].value_counts())
diagnosis_df

classification
broken_mapping    17
unexplained        1
Name: count, dtype: int64


,dataset,entity,name,reference,classification,config_assigned_curies,suggested_fix,suggested_curie,confidence,entity_url
0,central-activities-zone,2200001,,CAZ00000001,broken_mapping,local-organisation:LBH,local-organisation:LBH -> local-authority:LBH ...,local-authority:LBH,high,https://www.planning.data.gov.uk/entity/2200001
1,central-activities-zone,2200002,,CAZ00000002,broken_mapping,local-organisation:SWK,local-organisation:SWK -> local-authority:SWK ...,local-authority:SWK,high,https://www.planning.data.gov.uk/entity/2200002
2,central-activities-zone,2200003,,CAZ00000003,broken_mapping,local-organisation:WND,local-organisation:WND -> local-authority:WND ...,local-authority:WND,high,https://www.planning.data.gov.uk/entity/2200003
3,central-activities-zone,2200004,,CAZ00000004,broken_mapping,local-organisation:KEC,local-organisation:KEC -> local-authority:KEC ...,local-authority:KEC,high,https://www.planning.data.gov.uk/entity/2200004
4,central-activities-zone,2200005,,CAZ00000005,broken_mapping,local-organisation:WSM,local-organisation:WSM -> local-authority:WSM ...,local-authority:WSM,high,https://www.planning.data.gov.uk/entity/2200005
5,central-activities-zone,2200006,,CAZ00000006,broken_mapping,local-organisation:LND,local-organisation:LND -> local-authority:LND ...,local-authority:LND,high,https://www.planning.data.gov.uk/entity/2200006
6,central-activities-zone,2200007,,CAZ00000007,broken_mapping,local-organisation:CMD,local-organisation:CMD -> local-authority:CMD ...,local-authority:CMD,high,https://www.planning.data.gov.uk/entity/2200007
7,central-activities-zone,2200008,,CAZ00000008,broken_mapping,local-organisation:TWH,local-organisation:TWH -> local-authority:TWH ...,local-authority:TWH,high,https://www.planning.data.gov.uk/entity/2200008
8,central-activities-zone,2200009,,CAZ00000009,broken_mapping,local-organisation:ISL,local-organisation:ISL -> local-authority:ISL ...,local-authority:ISL,high,https://www.planning.data.gov.uk/entity/2200009
9,central-activities-zone,2200010,,CAZ00000010,broken_mapping,local-organisation:HCK,local-organisation:HCK -> local-authority:HCK ...,local-authority:HCK,high,https://www.planning.data.gov.uk/entity/2200010


## 8. Validate against endpoint provenance trace

The `broken_mapping`/`unexplained` suggestions above come from pattern-matching the config CSV text (prefix swaps, string similarity against sibling rows) -- plausible, but still inference, not proof. This traces `fact` → `fact_resource` → `log` → `endpoint` → `source` for **every one of the 18 entities** (not just ones classified `unassigned`) -- the same lineage `2026-07_correct_organisations_of_entities/4_full_population_provenance_trace.ipynb` uses to find who *actually* submitted an entity's data -- and checks whether the organisation actually registered against the entity's submitting endpoint agrees with what we suggested from config alone.

For any entity still classified `unassigned` after this (none currently), falls back to matching the endpoint URL's domain against `organisation.website` as a last resort.

In [9]:
import re


def _trace_true_org(dataset, entity):
    facts = datasette_sql(dataset, f"SELECT fact FROM fact WHERE entity = {entity}")
    if facts.empty:
        return None, None
    fr = datasette_sql(dataset, "SELECT fact, resource FROM fact_resource WHERE fact IN ({})".format(
        ",".join(f"'{f}'" for f in facts["fact"].unique())
    ))
    if fr.empty:
        return None, None
    log = datasette_sql("digital-land", "SELECT resource, endpoint FROM log WHERE resource IN ({})".format(
        ",".join(f"'{r}'" for r in fr["resource"].unique())
    )).drop_duplicates()
    if log.empty:
        return None, None
    endpoints = log["endpoint"].unique().tolist()
    source = datasette_sql("digital-land", "SELECT endpoint, organisation FROM source WHERE endpoint IN ({})".format(
        ",".join(f"'{e}'" for e in endpoints)
    ))
    endpoint_df = datasette_sql("digital-land", "SELECT endpoint, endpoint_url FROM endpoint WHERE endpoint IN ({})".format(
        ",".join(f"'{e}'" for e in endpoints)
    ))
    endpoint_url = endpoint_df["endpoint_url"].iloc[0] if not endpoint_df.empty else None
    if not source.empty:
        return source["organisation"].iloc[0], endpoint_url
    return None, endpoint_url


def _match_url_to_org(endpoint_url):
    if not endpoint_url:
        return None
    domain_match = re.search(r"https?://([^/]+)", endpoint_url)
    if not domain_match:
        return None
    domain = domain_match.group(1).lower().replace("www.", "")
    for curie, website in curie_to_website.items():
        if isinstance(website, str) and domain in website.lower():
            return curie
    return None


diagnosis_df["endpoint_trace_organisation"] = None
diagnosis_df["endpoint_url_traced"] = None
diagnosis_df["trace_agreement"] = None

for idx, row in diagnosis_df.iterrows():
    true_org, endpoint_url = _trace_true_org(row["dataset"], row["entity"])
    diagnosis_df.loc[idx, "endpoint_trace_organisation"] = true_org
    diagnosis_df.loc[idx, "endpoint_url_traced"] = endpoint_url

    if row["classification"] == "unassigned":
        if true_org:
            diagnosis_df.loc[idx, "suggested_fix"] = f"suggested org (source-registered): {true_org}"
            diagnosis_df.loc[idx, "suggested_curie"] = true_org
            diagnosis_df.loc[idx, "confidence"] = "high"
        else:
            url_guess = _match_url_to_org(endpoint_url)
            if url_guess:
                diagnosis_df.loc[idx, "suggested_fix"] = f"suggested org (URL text match against {endpoint_url}): {url_guess}"
                diagnosis_df.loc[idx, "suggested_curie"] = url_guess
                diagnosis_df.loc[idx, "confidence"] = "low"
            else:
                diagnosis_df.loc[idx, "suggested_fix"] = "no_data_to_suggest" + (f" (endpoint traced: {endpoint_url})" if endpoint_url else " (no endpoint traceable)")
        continue

    # broken_mapping / unexplained: does the actual submitting endpoint agree with our suggestion?
    if true_org is None:
        diagnosis_df.loc[idx, "trace_agreement"] = "no_endpoint_trace_available"
    elif row["suggested_curie"] is not None and true_org == row["suggested_curie"]:
        diagnosis_df.loc[idx, "trace_agreement"] = "agrees"
    else:
        diagnosis_df.loc[idx, "trace_agreement"] = f"traces_elsewhere:{true_org}"

# A "traces_elsewhere" result isn't automatically wrong -- if the traced organisation is
# ITSELF a registered source serving several different suggested_curie values in the same
# dataset, that's the signature of a regional aggregator (like GLA submitting London-wide
# central-activities-zone data on behalf of individual boroughs), which entity-organisation.csv
# exists specifically to override (see digital_land/phase/priority.py: the per-entity config
# assignment takes precedence over the submitting endpoint's own organisation). Only a
# traced org that DOESN'T show this aggregator pattern is a genuine red flag on our suggestion.
fanout = (
    diagnosis_df[diagnosis_df["trace_agreement"].astype(str).str.startswith("traces_elsewhere:")]
    .groupby(["dataset", "endpoint_trace_organisation"])["suggested_curie"]
    .nunique()
)
likely_aggregators = {org for (_, org), n in fanout.items() if n > 1}

for idx, row in diagnosis_df.iterrows():
    ta = row["trace_agreement"]
    if isinstance(ta, str) and ta.startswith("traces_elsewhere:"):
        traced_org = ta.split(":", 1)[1]
        if traced_org in likely_aggregators:
            diagnosis_df.loc[idx, "trace_agreement"] = (
                f"traces to {traced_org}, a likely regional aggregator "
                f"(registered active source serving {fanout[(row['dataset'], traced_org)]} different "
                f"suggested orgs here) -- consistent with entity-organisation.csv's per-entity override, not a red flag"
            )
        else:
            diagnosis_df.loc[idx, "trace_agreement"] = f"disagrees, no aggregator pattern -- worth a second look (endpoint traces to {traced_org})"

print(diagnosis_df["trace_agreement"].value_counts(dropna=False))
diagnosis_df[["dataset", "entity", "classification", "suggested_curie", "endpoint_trace_organisation", "trace_agreement"]]

trace_agreement
agrees                                                                                                                                                                                                             9
traces to local-authority:GLA, a likely regional aggregator (registered active source serving 9 different suggested orgs here) -- consistent with entity-organisation.csv's per-entity override, not a red flag    9
Name: count, dtype: int64


,dataset,entity,classification,suggested_curie,endpoint_trace_organisation,trace_agreement
0,central-activities-zone,2200001,broken_mapping,local-authority:LBH,local-authority:LBH,agrees
1,central-activities-zone,2200002,broken_mapping,local-authority:SWK,local-authority:GLA,"traces to local-authority:GLA, a likely region..."
2,central-activities-zone,2200003,broken_mapping,local-authority:WND,local-authority:GLA,"traces to local-authority:GLA, a likely region..."
3,central-activities-zone,2200004,broken_mapping,local-authority:KEC,local-authority:GLA,"traces to local-authority:GLA, a likely region..."
4,central-activities-zone,2200005,broken_mapping,local-authority:WSM,local-authority:GLA,"traces to local-authority:GLA, a likely region..."
5,central-activities-zone,2200006,broken_mapping,local-authority:LND,local-authority:GLA,"traces to local-authority:GLA, a likely region..."
6,central-activities-zone,2200007,broken_mapping,local-authority:CMD,local-authority:GLA,"traces to local-authority:GLA, a likely region..."
7,central-activities-zone,2200008,broken_mapping,local-authority:TWH,local-authority:GLA,"traces to local-authority:GLA, a likely region..."
8,central-activities-zone,2200009,broken_mapping,local-authority:ISL,local-authority:GLA,"traces to local-authority:GLA, a likely region..."
9,central-activities-zone,2200010,broken_mapping,local-authority:HCK,local-authority:GLA,"traces to local-authority:GLA, a likely region..."


## 9. Duplicate-entity check for `unexplained`/`unassigned` cases

Investigating the one `unexplained` case (`brownfield-land` entity 1742531, reference `BDB/63103`) found it isn't actually unexplained: `entity-organisation.csv` assigns a valid org, but the entity itself is a **duplicate** -- a brand-new resource (12 days old at the time of writing) from Basingstoke and Deane's own endpoint failed to match back to the long-standing canonical entity (1700711, same name/reference, coordinates ~15m apart, 8 resources back to 2018) and got a new entity ID minted instead, with a blank-organisation `lookup.csv` row.

Checked how widespread this specific pattern (`lookup.csv` row with blank `organisation`, whose `(prefix, reference)` is shared with a *different* entity that does have an `organisation`) is across all 55 pipelines: **1,950 historical rows, but confined to just 2 pipelines** (`listed-building`: 1,892, `brownfield-land`: 58) -- and almost all of it has already self-healed. Cross-checked against the live platform: of those ~1,950, only entity 1742531 itself still exists as a live entity at all; everything else points to entity IDs that have since been retired/redirected (`listed-building`'s survivors already have `organisation_entity` populated via a different route entirely). So this is a real, recurring pipeline mechanism -- worth flagging to Swati as its own observation -- but normally transient, and not currently a live contributor to the missing-`organisation_entity` problem beyond this one case.

This section runs that same duplicate check for any entity still classified `unexplained` or `unassigned`, to reclassify with a concrete answer where the pattern applies.

In [10]:
def _find_duplicate_entity_explanation(dataset, entity, reference):
    lookup_df = fetch_config_csv(dataset, "lookup")
    if lookup_df.empty or "reference" not in lookup_df.columns:
        return None
    lookup_df = lookup_df.copy()
    lookup_df["reference"] = lookup_df["reference"].astype(str)
    same_ref = lookup_df[lookup_df["reference"] == str(reference)]
    if len(same_ref) < 2:
        return None

    other_rows = same_ref[same_ref["entity"].astype("Int64") != entity]
    candidates = other_rows[
        other_rows["organisation"].notna() & (other_rows["organisation"].astype(str).str.strip() != "")
    ]
    if candidates.empty:
        return None

    candidate_entities = candidates["entity"].astype(int).unique().tolist()
    live = datasette_sql(dataset, "SELECT entity FROM entity WHERE entity IN ({})".format(
        ",".join(str(e) for e in candidate_entities)
    ))
    still_live = live["entity"].tolist() if not live.empty else []
    if not still_live:
        return None

    canonical_org = candidates[candidates["entity"].astype(int).isin(still_live)]["organisation"].iloc[0]
    return still_live[0], canonical_org


for idx, row in diagnosis_df.iterrows():
    if row["classification"] not in ("unexplained", "unassigned"):
        continue
    result = _find_duplicate_entity_explanation(row["dataset"], int(row["entity"]), row["reference"])
    if result is None:
        continue
    canonical_entity, canonical_org = result
    diagnosis_df.loc[idx, "classification"] = "likely_duplicate_entity"
    diagnosis_df.loc[idx, "suggested_fix"] = (
        f"appears to be a duplicate of entity {canonical_entity} (org {canonical_org}) -- same reference "
        f"'{row['reference']}' already has a canonical, org-assigned lookup.csv entry for a different, "
        f"still-live entity; likely created by a lookup-match failure on a later resubmission (see README)"
    )
    diagnosis_df.loc[idx, "suggested_curie"] = canonical_org
    diagnosis_df.loc[idx, "confidence"] = "medium"

print(diagnosis_df["classification"].value_counts())
diagnosis_df[diagnosis_df["classification"] == "likely_duplicate_entity"][
    ["dataset", "entity", "reference", "classification", "suggested_fix"]
]

classification
broken_mapping             17
likely_duplicate_entity     1
Name: count, dtype: int64


,dataset,entity,reference,classification,suggested_fix
17,brownfield-land,1742531,BDB/63103,likely_duplicate_entity,appears to be a duplicate of entity 1700711 (o...


## 10. How widespread is the duplicate-entity mechanism, platform-wide?

Section 9 explained one case (`brownfield-land` entity 1742531) via a specific pattern: a `lookup.csv` row with a blank `organisation`, whose `(prefix, reference)` is shared with a *different* entity that does have one -- the fingerprint of a resubmission that failed to match its long-standing entity and got a new one minted instead. This checks how common that pattern is across all 55 `lookup.csv` files, and -- since a duplicate entity like this often gets reconciled/retired on a later pipeline run -- cross-checks how many of those historical occurrences are still live entities today, to see whether this is an active, ongoing problem or a mostly self-healing one.

In [11]:
def _blank_ambiguous_entities_for_pipeline(dataset):
    lookup_df = fetch_config_csv(dataset, "lookup")
    if lookup_df.empty or not {"prefix", "reference", "organisation", "entity"}.issubset(lookup_df.columns):
        return []
    ref_rows = lookup_df[lookup_df["reference"].notna()].copy()
    if ref_rows.empty:
        return []
    ref_rows["organisation"] = ref_rows["organisation"].fillna("").astype(str).str.strip()
    ref_rows["key"] = list(zip(ref_rows["prefix"], ref_rows["reference"]))

    # a (prefix, reference) used by more than one distinct entity is genuinely ambiguous --
    # not just reused casually, but actually pointing at different records
    ambiguous_keys = set(ref_rows.groupby("key")["entity"].nunique().loc[lambda s: s > 1].index)
    # only count it if some OTHER row for that same key does carry an organisation -- i.e. a
    # qualifier clearly exists and would disambiguate it, this row just lacks it
    has_org_keys = set(ref_rows.loc[ref_rows["organisation"] != "", "key"])

    hits = ref_rows[
        (ref_rows["organisation"] == "") & ref_rows["key"].isin(ambiguous_keys) & ref_rows["key"].isin(has_org_keys)
    ]
    return hits["entity"].astype(int).tolist()


print(f"Scanning lookup.csv across all {len(lookup_pipelines)} pipelines for the duplicate-entity pattern...")
pattern_hits = {}
for dataset in lookup_pipelines:
    entities = _blank_ambiguous_entities_for_pipeline(dataset)
    if entities:
        pattern_hits[dataset] = entities

total_historical = sum(len(v) for v in pattern_hits.values())
print(f"{total_historical} historical blank-and-ambiguous lookup.csv rows across {len(pattern_hits)} of {len(lookup_pipelines)} pipelines")

# Cross-check against live data: of those historical rows, how many are still live entities
# at all, and of those, how many are still blank -- i.e. how much of this has self-healed.
self_heal_rows = []
for dataset, entities in pattern_hits.items():
    live_parts = [
        datasette_sql(dataset, "SELECT entity, organisation_entity FROM entity WHERE entity IN ({})".format(
            ",".join(str(e) for e in batch)
        ))
        for batch in chunk(entities, 300)
    ]
    live = pd.concat(live_parts, ignore_index=True) if live_parts else pd.DataFrame(columns=["entity", "organisation_entity"])
    self_heal_rows.append({
        "dataset": dataset,
        "historical_rows": len(entities),
        "still_live": len(live),
        "still_live_and_blank_org": live["organisation_entity"].isna().sum() if not live.empty else 0,
    })

self_heal_df = pd.DataFrame(self_heal_rows).sort_values("historical_rows", ascending=False)
print(f"\n{self_heal_df['still_live_and_blank_org'].sum()} of those {total_historical} historical rows are both still live AND still blank today -- the rest have self-healed (retired/redirected, or resolved via another route)")
self_heal_df

Scanning lookup.csv across all 55 pipelines for the duplicate-entity pattern...


/Users/sianteesdale/Documents/GitHub/jupyter-analysis/analysis/2026-09_data-quality-ticket-checks/helpers.py:48: DtypeWarning: Columns (1,4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(StringIO(r.text))


1987 historical blank-and-ambiguous lookup.csv rows across 2 of 55 pipelines



1 of those 1987 historical rows are both still live AND still blank today -- the rest have self-healed (retired/redirected, or resolved via another route)


,dataset,historical_rows,still_live,still_live_and_blank_org
1,listed-building,1929,1891,0
0,brownfield-land,58,1,1


## 11. Export

In [12]:
# One file, one row per affected entity -- the dataset-level summary and the config-repo-wide
# audit are useful checkpoints along the way (see the printed output in sections 2 and 6 above)
# but both are fully derivable from this single entity-level file, so they don't need their own
# separate exports. local-plan-document / local-plan-timetable excluded entirely -- confirmed
# stale, superseded by development-plan-document / development-plan-timetable (section 5), so
# their gaps aren't real, current data-quality issues worth acting on.
diagnosis_cols = diagnosis_df[[
    "dataset", "entity", "classification", "config_assigned_curies", "suggested_fix",
    "suggested_curie", "confidence", "endpoint_trace_organisation", "endpoint_url_traced", "trace_agreement",
]]

legacy_datasets = ["local-plan-document", "local-plan-timetable"]
final_df = missing_entities_df[~missing_entities_df["dataset"].isin(legacy_datasets)].merge(
    diagnosis_cols, on=["dataset", "entity"], how="left"
)

out_path = os.path.join(DATA_DIR, "missing_organisation_entity.csv")
final_df.to_csv(out_path, index=False)
print(f"Saved {len(final_df)} rows to {out_path} ({len(missing_entities_df) - len(final_df)} legacy-dataset rows excluded)")

Saved 18 rows to ../../data/missing_organisation_entity.csv (90 legacy-dataset rows excluded)
